In [7]:
import pandas as pd
import ast
from pathlib import Path
from dataclasses import dataclass
from typing import Set, Optional

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "jobs_featured_cleaned.csv"

df = pd.read_csv(DATA_PATH)

In [2]:
@dataclass
class CandidateProfile:
    """
    User profile used for job matching.
    """

    skills: Set[str]

    experience_years: int

    city: str

    remote_preference: bool = False

    internship_preference: bool = False

    minimum_salary: Optional[float] = None

## Skill Match Score (Jaccard Similarity)
To measure the similarity between a candidate's skills and the required skills of a job, we use **Jaccard Similarity**.
Jaccard Similarity measures the similarity between two sets by comparing their intersection and union.
In this context:
- **A = Candidate skills (`candidate_skills`)**
- **B = Required job skills (`job_skills`)**
The formula:
$$
J(A,B) = \frac{|A \cap B|}{|A \cup B|}
$$
Where:
- $A \cap B$ represents the **common skills** shared between the candidate and the job.
- $A \cup B$ represents the **total unique skills** across both sets.
The score ranges from **0 to 1**:
- **0** → No matching skills between candidate and job.
- **1** → Candidate skills and job requirements are exactly the same.
### Example
Candidate Skills:


In [3]:
def calculate_skill_match(candidate_skills: set, job_skills: set) -> float:
    """
    Calculate skill similarity between candidate skills and job requirements
    using Jaccard Similarity.
    Formula:
        J(A,B) = |A ∩ B| / |A ∪ B|
    Where:
        A ∩ B -> Common skills between candidate and job
        A ∪ B -> Total unique skills across both sets
    Returns:
        float: Similarity score between 0 and 1.
    """

    if not job_skills:
        return 0.0

    intersection = candidate_skills & job_skills
    union = candidate_skills | job_skills

    return len(intersection) / len(union)


candidate_skills = {"python", "docker", "postgresql"}
job_skills = {"python", "docker", "linux", "postgresql"}

calculate_skill_match(candidate_skills, job_skills)

0.75

In [6]:
type(df["required_skills_set"].iloc[0]), df["required_skills_set"].iloc[0]

(str, "{'oracle_database', 'python', 'git'}")

In [9]:
def restore_skill_set(value):
    """
    Convert string representation of set back to Python set.
    """
    if isinstance(value, set):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    return set()

df["required_skills_set"] = df["required_skills_set"].apply(restore_skill_set)

In [11]:
candidate = CandidateProfile(
    skills={
        "python",
        "docker",
        "postgresql"
    },
    experience_years=3,
    city="تهران",
    remote_preference=True
)

df["skill_match_score"] = df["required_skills_set"].apply(lambda x: calculate_skill_match(candidate.skills, x))

In [12]:
df["skill_match_score"].describe()

count    12037.000000
mean         0.014739
std          0.060011
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.750000
Name: skill_match_score, dtype: float64

## Job Skill Coverage Score
The main question behind this feature is:
**"What percentage of the job requirements does the candidate satisfy?"**
Unlike Jaccard Similarity, which measures the overlap between two skill sets relative to their union, Job Skill Coverage focuses only on the required skills of the job.
The formula:
$$
Job\ Coverage = \frac{|Candidate\ Skills \cap Job\ Skills|}{|Job\ Required\ Skills|}
$$
Where:
- **Candidate Skills ∩ Job Skills** → Skills that are common between the candidate and the job requirements (**Matched Skills**)
- **Job Required Skills** → Total number of skills required by the job
The score ranges from **0 to 1**:
- **0** → Candidate does not have any of the required skills.
- **1** → Candidate has all required skills.
### Example
Candidate Skills:

In [ ]:
def calculate_skill_coverage(candidate_skills: set, job_skills: set) -> float:
    """
    Percentage of required job skills covered by candidate.
    """
    if not job_skills:
        return 0.0
    matched_skills = (candidate_skills & job_skills)
    return len(matched_skills) / len(job_skills)

df["skill_coverage_score"] = df["required_skills_set"].apply(lambda x: calculate_skill_coverage(candidate.skills, x))

In [15]:
df[["title", "company", "required_skills_set", "skill_coverage_score"]].sort_values("skill_coverage_score", ascending=False)[:20]

,title,company,required_skills_set,skill_coverage_score
59,کارشناس نرم افزار,ایتوک ایران,{python},1.0
614,طراح سایت,موبولند,{python},1.0
398,تحلیل گر بازار سرمایه,گسترش اقتصادی زمرد,{python},1.0
371,Senior Tier 2 SOC Analyst - آقا,امن پردازان کویر,{python},1.0
52,Senior Python Developer,گروه حمل و نقل بین المللی ادمیرال,{python},1.0
660,Backend Developer,مهسان,{docker},1.0
621,پژوهشگر دکتری شیمی و فیزیک,تحقیق و توسعه تک فن,{python},1.0
164,برنامه‌نویس نرم‌افزار – سطح Junior,گسترش فناوری های نوین,{python},1.0
98,کارآموز هوش مصنوعی و پایتون,هوش افزار نوآفرین,{python},1.0
244,Machine Learning Engineer,نئوبانک اقتصاد نوین,{python},1.0


In [18]:
def calculate_experience_match(candidate_experience: int, job_experience: int) -> float:
    """
    Compare candidate experience with required job experience.
    """
    difference = abs(candidate_experience - job_experience)
    score = 1 / (1 + difference)
    return score

calculate_experience_match(3, 3)

1.0

In [19]:
calculate_experience_match(3, 5)

0.3333333333333333

In [20]:
df["experience_match_score"] = df["experience_years"].apply(lambda x: calculate_experience_match(candidate.experience_years, x))

In [22]:
df[["title", "experience_years", "experience_match_score"]].sort_values("experience_match_score", ascending=False)[:20]

,title,experience_years,experience_match_score
8,توسعه‌دهنده Python,3,1.0
12036,مسئول سایت و شبکه - آقا,3,1.0
12035,مدیر محصول,3,1.0
1,Senior Backend Engineer(Python/FastAPI)- Shoraka,3,1.0
10,برنامه‌نویس بک‌اند (Python),3,1.0
12,برنامه نویس پایتون,3,1.0
6809,کارشناس حسابداری - آقا,3,1.0
6811,مدیر کنترل کیفیت - آقا,3,1.0
6790,کارشناس حقوق و دستمزد,3,1.0
6791,کارشناس ارشد بازاریابی,3,1.0


In [23]:
def calculate_location_match(candidate_city: str, job_city: str) -> float:
    """
    Calculate location compatibility.
    """
    if candidate_city == job_city:
        return 1.0
    return 0.0

df["location_match_score"] = df["city"].apply(lambda x: calculate_location_match(candidate.city, x))

In [25]:
df[["title", "city", "location_match_score"]].sort_values("location_match_score", ascending=False)[:20]

,title,city,location_match_score
12036,مسئول سایت و شبکه - آقا,تهران,1.0
0,برنامه نویس Python,تهران,1.0
1,Senior Backend Engineer(Python/FastAPI)- Shoraka,تهران,1.0
2,برنامه نویس Python,تهران,1.0
3,برنامه نویس بک‌اند (Python/Django),تهران,1.0
12020,کارشناس سیویل و سازه,تهران,1.0
12019,منشی و مسئول دفتر,تهران,1.0
12017,فروشنده و صندوقدار - خانم,تهران,1.0
12016,کارشناس مهندسی مکانیک / متالوژی - آقا,تهران,1.0
12015,طراح گرافیک - خانم,تهران,1.0


In [27]:
def calculate_remote_match(candidate_remote_preference: bool, job_remote: bool) -> float:
    """
    Match candidate remote preference with job option.
    """
    if not candidate_remote_preference:
        return 1.0
    if job_remote:
        return 1.0
    return 0.3

df["remote_match_score"] = df["is_remote"].apply(lambda x: calculate_remote_match(candidate.remote_preference, x))

In [28]:
df[["title", "remote", "remote_match_score"]].sample(20, random_state=42)

,title,remote,remote_match_score
9432,اپراتور و تکنسین CNC (ماشین ابزار) - آقا,False,0.3
10171,تحلیلگر محتوا (بازارهای مالی),True,1.0
487,پشتیبان فنی فروش - خانم,False,0.3
10890,کارشناس تولید محتوا و شبکه های اجتماعی,False,0.3
8688,کارشناس بازاریابی و فروش,False,0.3
1483,کارشناس طراحی الکترونیک,False,0.3
11575,مقاله‌نویس - خانم,False,0.3
2627,مدیر مالی و حسابداری,False,0.3
3777,توسعه‌دهنده جاوا,False,0.3
6562,کارشناس تولید,False,0.3


In [30]:
def calculate_final_score(row):
    """
    Combine different matching signals into one ranking score.
    """
    score = (
        0.5 * row["skill_coverage_score"]
        +
        0.2 * row["experience_match_score"]
        +
        0.15 * row["location_match_score"]
        +
        0.1 * row["remote_match_score"]
    )
    return score

df["final_match_score"] = df.apply(calculate_final_score, axis=1)

In [31]:
df[[
        "title",
        "company",
        "skill_coverage_score",
        "experience_match_score",
        "location_match_score",
        "remote_match_score",
        "final_match_score"
    ]
].sort_values(
    "final_match_score",
    ascending=False
)[:20]

,title,company,skill_coverage_score,experience_match_score,location_match_score,remote_match_score,final_match_score
168,کارشناس ارشد هوش مصنوعی (AI & Process Optimization Specialist),ایکس چیف,1.0,1.000000,1.0,1.0,0.950000
470,مدیر محصول,گرایش تازه کیش,1.0,1.000000,1.0,0.3,0.880000
125,تحلیلگر داده امنیت (Security Data Analyst),فناوری اطلاعات و ارتباطات مُهَیمن,1.0,1.000000,1.0,0.3,0.880000
141,توسعه‌دهنده عامل هوش مصنوعی (AI Agent Developer),شرکت داده و اعتبار سنجی تجارت ایرانیان (داتا),1.0,1.000000,1.0,0.3,0.880000
267,Blockchain Developer,آبان تتر,1.0,1.000000,1.0,0.3,0.880000
488,Senior Product Data Analyst,گروه اسنپ,1.0,1.000000,1.0,0.3,0.880000
76,برنامه‌ نویس هوش مصنوعی,سینوکس,1.0,0.333333,1.0,1.0,0.816667
405,کارشناس هوش مصنوعی,گروه ساختمانی R1 ( آروان),1.0,0.250000,1.0,1.0,0.800000
680,Senior Software Engineer (Go),اسنپ ساپلای,1.0,0.500000,1.0,0.3,0.780000
660,Backend Developer,مهسان,1.0,0.500000,1.0,0.3,0.780000


In [33]:
def calculate_skill_strength(candidate_skills:set, job_skills:set):
    if not candidate_skills:
        return 0
    matched = candidate_skills & job_skills
    return len(matched) / len(candidate_skills)

df["skill_strength_score"] = df["required_skills_set"].apply(lambda x: calculate_skill_strength(candidate.skills, x))

In [34]:
def calculate_final_score(row):
    return (
        0.35 * row["skill_coverage_score"]
        +
        0.20 * row["skill_strength_score"]
        +
        0.15 * row["experience_match_score"]
        +
        0.15 * row["location_match_score"]
        +
        0.10 * row["remote_match_score"]
    )

In [35]:
df.sort_values("final_match_score", ascending=False)[[
"title",
"required_skills_set",
"skill_coverage_score",
"skill_strength_score",
"final_match_score"
]
][:20]

,title,required_skills_set,skill_coverage_score,skill_strength_score,final_match_score
168,کارشناس ارشد هوش مصنوعی (AI & Process Optimization Specialist),{python},1.0,0.333333,0.950000
470,مدیر محصول,{python},1.0,0.333333,0.880000
125,تحلیلگر داده امنیت (Security Data Analyst),{python},1.0,0.333333,0.880000
141,توسعه‌دهنده عامل هوش مصنوعی (AI Agent Developer),"{docker, python}",1.0,0.666667,0.880000
267,Blockchain Developer,{python},1.0,0.333333,0.880000
488,Senior Product Data Analyst,{python},1.0,0.333333,0.880000
76,برنامه‌ نویس هوش مصنوعی,{python},1.0,0.333333,0.816667
405,کارشناس هوش مصنوعی,{python},1.0,0.333333,0.800000
680,Senior Software Engineer (Go),{postgresql},1.0,0.333333,0.780000
660,Backend Developer,{docker},1.0,0.333333,0.780000


In [38]:
candidate.target_roles = {
    "python developer",
    "backend developer",
    "machine learning engineer",
    "data analyst",
    "ai engineer"
}

def calculate_title_match(job_title, target_titles):
    title = job_title.lower()
    for target in target_titles:
        if target.lower() in title:
            return 1.0
    return 0.0

candidate_titles = [
    "python",
    "backend",
    "machine learning",
    "data analyst",
    "هوش مصنوعی"
]


df["title_match_score"] = df["title"].apply(lambda x: calculate_title_match(x, candidate_titles))

In [43]:
def calculate_final_score(row):
    return (
        0.30 * row["skill_coverage_score"]
        +
        0.15 * row["skill_strength_score"]
        +
        0.20 * row["title_match_score"]
        +
        0.15 * row["experience_match_score"]
        +
        0.10 * row["location_match_score"]
        +
        0.10 * row["remote_match_score"]
    )
df["final_match_score"] = df.apply(calculate_final_score, axis=1)

In [44]:
df[
[
"title",
"company",
"skill_coverage_score",
"skill_strength_score",
"title_match_score",
"final_match_score"
]
].sort_values(
    "final_match_score",
    ascending=False
)[:20]

,title,company,skill_coverage_score,skill_strength_score,title_match_score,final_match_score
168,کارشناس ارشد هوش مصنوعی (AI & Process Optimization Specialist),ایکس چیف,1.000000,0.333333,1.0,0.900000
141,توسعه‌دهنده عامل هوش مصنوعی (AI Agent Developer),شرکت داده و اعتبار سنجی تجارت ایرانیان (داتا),1.000000,0.666667,1.0,0.880000
96,مهندس ارشد هوش مصنوعی و بک‌اند,قهوه ارگن,0.500000,1.000000,1.0,0.850000
125,تحلیلگر داده امنیت (Security Data Analyst),فناوری اطلاعات و ارتباطات مُهَیمن,1.000000,0.333333,1.0,0.830000
488,Senior Product Data Analyst,گروه اسنپ,1.000000,0.333333,1.0,0.830000
1,Senior Backend Engineer(Python/FastAPI)- Shoraka,بیمه بازار,0.428571,1.000000,1.0,0.828571
30,برنامه‌نویس Python/Odoo,گروه بازرگانی بی نی سی,0.600000,1.000000,1.0,0.810000
338,کارشناس توسعه دهنده هوش مصنوعی,شرکت خدمات انفورماتیک,1.000000,0.666667,1.0,0.805000
76,برنامه‌ نویس هوش مصنوعی,سینوکس,1.000000,0.333333,1.0,0.800000
405,کارشناس هوش مصنوعی,گروه ساختمانی R1 ( آروان),1.000000,0.333333,1.0,0.787500


In [45]:
def calculate_skill_complexity(job_skills):
    if not job_skills:
        return 0
    return min(len(job_skills) / 5, 1)

df["skill_complexity_score"] = df["required_skills_set"].apply(calculate_skill_complexity)

In [47]:
def calculate_final_score(row):
    return (
        0.25 * row["skill_coverage_score"]
        +
        0.15 * row["skill_strength_score"]
        +
        0.10 * row["skill_complexity_score"]
        +
        0.20 * row["title_match_score"]
        +
        0.15 * row["experience_match_score"]
        +
        0.10 * row["location_match_score"]
        +
        0.05 * row["remote_match_score"]
    )
df["final_match_score"] = df.apply(calculate_final_score, axis=1)

In [ ]:
top_jobs = df.sort_values("final_match_score", ascending=False)[:20]
top_jobs[
[
"title",
"company",
"required_skills_set",
"skill_coverage_score",
"skill_strength_score",
"skill_complexity_score",
"title_match_score",
"final_match_score"
]
]

,title,company,required_skills_set,skill_coverage_score,skill_strength_score,skill_complexity_score,title_match_score,final_match_score
96,مهندس ارشد هوش مصنوعی و بک‌اند,قهوه ارگن,"{postgresql, next.js, docker, python, mongodb, git}",0.500000,1.000000,1.0,1.0,0.875000
30,برنامه‌نویس Python/Odoo,گروه بازرگانی بی نی سی,"{postgresql, docker, python, javascript, git}",0.600000,1.000000,1.0,1.0,0.865000
1,Senior Backend Engineer(Python/FastAPI)- Shoraka,بیمه بازار,"{gerafana, postgresql, prometheus, docker, python, kubernetes, redis}",0.428571,1.000000,1.0,1.0,0.857143
141,توسعه‌دهنده عامل هوش مصنوعی (AI Agent Developer),شرکت داده و اعتبار سنجی تجارت ایرانیان (داتا),"{docker, python}",1.000000,0.666667,0.4,1.0,0.855000
90,مهندس هوش مصنوعی و نرم‌افزار,نهادین آرمان,"{mysql, rest_api, postgresql, docker, python, django, git}",0.428571,1.000000,1.0,1.0,0.822143
17,برنامه‌نویس Back-End (Python),پارس تصمیم,"{rabbitmq, rest_api, postgresql, docker, python, mongodb, git}",0.428571,1.000000,1.0,1.0,0.822143
168,کارشناس ارشد هوش مصنوعی (AI & Process Optimization Specialist),ایکس چیف,{python},1.000000,0.333333,0.2,1.0,0.820000
8,توسعه‌دهنده Python,پارادایس هاب,"{mysql, postgresql, docker, python, mongodb, redis, django, git}",0.375000,1.000000,1.0,1.0,0.808750
60,Senior Backend Developer (Python),شرکت توسعه فن آوری هوش مصنوعی تفاهم,"{gerafana, rest_api, postgresql, prometheus, docker, python, redis, django, git}",0.333333,1.000000,1.0,1.0,0.798333
488,Senior Product Data Analyst,گروه اسنپ,{python},1.000000,0.333333,0.2,1.0,0.785000


In [ ]:
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / "jobs_match_scored.csv"
df.to_csv(OUTPUT_DIR / "jobs_match_scored.csv", index=False, encoding="utf-8-sig")
print(f"Dataset saved to: {output_path}")

Dataset saved to: ..\data\processed\jobs_match_scored.csv
